# Verified Phase 2 lineage — Natural Sampling Phase 1

This notebook belongs to the corrected AOI-masked Phase 2 workflow. It must use only:

- `Models/Phase2_Harmonized_GEDIAnchored_NaturalP1` for Phase 2 checkpoints;
- `Results/Final_Article_Harmonized_GEDIAnchored_NaturalP1` for evaluation products;
- `Inference_Harmonized_GEDIAnchored_NaturalP1` for annual maps.

The former `Phase2_AOI_Masked`, `Final_Article_AOI_Masked`, and `Inference_AOI_Masked` products were generated from an incorrect Phase 1 parent lineage and must not be used. Run `Phase_2.ipynb` first, followed by `Inference.ipynb`, before regenerating downstream figures.


> **Active lineage (2026-08-09).** This notebook uses the harmonised GEDI-anchored Phase 2 checkpoints. Training sequences contain at least one valid GEDI observation, while dense image-only sequences are reserved for wall-to-wall inference. Execute the notebook from the first cell; outputs from the former AOI-only Phase 2 lineage are not reused.


# Natural Sampling publication pipeline

This notebook is the isolated Natural Sampling copy. The original official pipeline remains unchanged. Training is disabled because the selected Phase 1 and Phase 2 models are already archived under `Natural_Sampling/Models`. Run the notebook from the first cell to regenerate figures and tables under `Natural_Sampling/Results`.


# Phase 2 — article plots for the three Moroccan forest ecosystems

This notebook creates **plots only** from the frozen, unique-nearest TEST
predictions of the three selected AOI-masked Phase 2 GrowthLoss models. It does not train,
select checkpoints, or modify predictions.

Evaluation domains:

| Ecosystem | Forest | TEST domain |
|---|---|---:|
| Moderately dense | Ifran | GEDI RH95 2–45 m |
| Low density | Maamoura | GEDI RH95 2–20 m |
| Sparse | Agadir | GEDI RH95 2–20 m |

Every figure is exported as PNG (1200 dpi), SVG and PDF. Long scientific titles
are intentionally omitted so that titles and captions can be written flexibly
in LaTeX.

Height-stratified plots use 5 m classes. For Maamoura and Agadir, the final displayed class is 10–15 m; observations above 15 m remain included in the global 2–20 m metrics and scatter plots.


In [ ]:
# 1 — Imports, immutable paths and TEST preflight
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
PHASE2_RESULTS = PROJECT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "Phase2"
OUTPUT_ROOT = PHASE2_RESULTS / "Article_Plots_Three_Forests"

SITES = {
    "Ifran": {
        "ecosystem": "Moderately Dense",
        "input": PHASE2_RESULTS / "Ifran" / "test_unique_nearest.csv.gz",
        "eval_min": 2.0,
        "eval_max": 45.0,
        "height_bins": np.asarray([0.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0, 45.0]),
        "expected_n": 5076,
        "phase1_sha256": "072a8735973e3511564cf3ef7907f5b33105df08895c78917fb817de6198afb1",
    },
    "Maamoura": {
        "ecosystem": "Low Density",
        "input": PHASE2_RESULTS / "Maamoura" / "test_unique_nearest.csv.gz",
        "eval_min": 2.0,
        "eval_max": 20.0,
        "height_bins": np.asarray([0.0, 5.0, 10.0, 15.0, 20.0]),
        "expected_n": 1799,
        "phase1_sha256": "d83a5493715451a61c997a66a25f361080a56e7ceade60eb886f2ae76f6bd7f4",
    },
    "Agadir": {
        "ecosystem": "Sparse",
        "input": PHASE2_RESULTS / "Agadir" / "test_unique_nearest.csv.gz",
        "eval_min": 2.0,
        "eval_max": 20.0,
        "height_bins": np.asarray([0.0, 5.0, 10.0, 15.0, 20.0]),
        "expected_n": 6125,
        "phase1_sha256": "f72f06e33455852cbc94cd00ecb81e192f68e7f67b89de007fef4a610fa25b2e",
    },
}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


REQUIRED_COLUMNS = {
    "split", "aux_shot_uid", "rh95", "pred_on_growthloss",
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

preflight_rows = []
for forest, cfg in SITES.items():
    path = cfg["input"]
    if not path.is_file():
        raise FileNotFoundError(
            f"{forest}: frozen Phase 2 TEST predictions not found: {path}\n"
            "Run the final Phase 2 workflow before executing this plots-only notebook."
        )
    lineage_path = path.parent / "lineage.json"
    if not lineage_path.is_file():
        raise FileNotFoundError(f"{forest}: missing Phase 2 lineage: {lineage_path}")
    lineage = json.loads(lineage_path.read_text(encoding="utf-8"))
    if lineage.get("phase1_parent_sha256") != cfg["phase1_sha256"]:
        raise RuntimeError(f"{forest}: incorrect Natural Sampling Phase 1 parent lineage")
    registry = json.loads((PROJECT / "Source" / "Project" / "final_selected_phase2_models.json").read_text(encoding="utf-8"))["models"]
    selected = registry[forest.lower()]
    checkpoint = Path(lineage.get("checkpoint", ""))
    if (not checkpoint.is_file() or file_sha256(checkpoint) != lineage.get("checkpoint_sha256")
            or lineage.get("checkpoint_sha256") != selected["checkpoint_sha256"]):
        raise RuntimeError(f"{forest}: Phase 2 checkpoint hash does not match lineage.json")
    header = pd.read_csv(path, nrows=2)
    missing = REQUIRED_COLUMNS - set(header.columns)
    if missing:
        raise RuntimeError(f"{forest}: missing columns {sorted(missing)} in {path}")
    if int(lineage.get("n", -1)) != cfg["expected_n"]:
        raise RuntimeError(f"{forest}: lineage n={lineage.get('n')} != expected {cfg['expected_n']}")
    preflight_rows.append({
        "forest": forest,
        "ecosystem": cfg["ecosystem"],
        "input": str(path),
        "eval_domain_m": f'{cfg["eval_min"]:g}–{cfg["eval_max"]:g}',
        "phase1_parent_sha256": lineage["phase1_parent_sha256"],
        "phase2_product": selected["product"],
        "lambda_temp": float(selected["lambda_temp"]),
        "D_m": float(selected["drop_m"]),
        "K": int(selected["K"]),
        "phase2_checkpoint_sha256": lineage["checkpoint_sha256"],
        "n": lineage["n"],
        "status": "PASS",
    })

preflight = pd.DataFrame(preflight_rows)
display(preflight)
print("Plots output:", OUTPUT_ROOT)


In [ ]:
# 2 — Shared loading, metrics, plotting and export functions
GEDI_HIST = "lightsteelblue"
GEDI_BAR = "steelblue"
PRED_COLOR = "darkorange"
MEDIAN_COLOR = "red"
ACCENT_BLUE = "#1f77b4"
ACCENT_ORANGE = "#ff7f0e"

# Publication typography: axis labels and tick numerals use exactly the same
# font family, weight and size on primary, secondary and colorbar axes.
AXIS_FONT_FAMILY = "DejaVu Sans"
AXIS_TEXT_SIZE = 11
plt.rcParams.update({
    "font.family": AXIS_FONT_FAMILY,
    "font.size": AXIS_TEXT_SIZE,
    "axes.labelsize": AXIS_TEXT_SIZE,
    "axes.labelweight": "normal",
    "xtick.labelsize": AXIS_TEXT_SIZE,
    "ytick.labelsize": AXIS_TEXT_SIZE,
})


def harmonize_axis_typography(*axes):
    """Apply one publication font to axis labels and all tick numerals."""
    for axis in axes:
        if axis is None:
            continue
        axis.xaxis.label.set_fontfamily(AXIS_FONT_FAMILY)
        axis.yaxis.label.set_fontfamily(AXIS_FONT_FAMILY)
        axis.xaxis.label.set_fontsize(AXIS_TEXT_SIZE)
        axis.yaxis.label.set_fontsize(AXIS_TEXT_SIZE)
        axis.tick_params(axis="both", which="both", labelsize=AXIS_TEXT_SIZE)
        for label in axis.get_xticklabels() + axis.get_yticklabels():
            label.set_fontfamily(AXIS_FONT_FAMILY)
            label.set_fontweight("normal")


def export_figure(fig, output_dir: Path, stem: str):
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for suffix in ("png", "svg", "pdf"):
        path = output_dir / f"{stem}.{suffix}"
        fig.savefig(
            path,
            dpi=1200 if suffix == "png" else None,
            bbox_inches="tight",
            pad_inches=0.03,
            facecolor="white",
        )
        paths.append(path)
    print("Saved:", ", ".join(str(path) for path in paths))
    return paths


def load_phase2_test(forest: str) -> pd.DataFrame:
    cfg = SITES[forest]
    frame = pd.read_csv(cfg["input"])
    if not frame["split"].astype(str).str.lower().eq("test").all():
        raise AssertionError(f"{forest}: non-TEST rows found")
    if frame["aux_shot_uid"].astype(str).duplicated().any():
        raise AssertionError(f"{forest}: duplicated aux_shot_uid after unique-nearest")
    frame["rh95"] = pd.to_numeric(frame["rh95"], errors="coerce")
    frame["prediction"] = pd.to_numeric(
        frame["pred_on_growthloss"], errors="coerce"
    )
    valid = (
        np.isfinite(frame["rh95"])
        & np.isfinite(frame["prediction"])
        & frame["rh95"].between(cfg["eval_min"], cfg["eval_max"], inclusive="both")
    )
    frame = frame.loc[valid].copy().reset_index(drop=True)
    if frame.empty:
        raise RuntimeError(f"{forest}: no valid Phase 2 TEST observation")
    if len(frame) != cfg["expected_n"]:
        raise RuntimeError(f"{forest}: valid TEST n={len(frame)} != expected {cfg['expected_n']}")
    return frame


def compute_metrics(frame: pd.DataFrame) -> dict:
    true = frame["rh95"].to_numpy(float)
    pred = frame["prediction"].to_numpy(float)
    error = pred - true
    sst = float(np.sum((true - true.mean()) ** 2))
    true_std = float(np.std(true, ddof=0))
    pred_std = float(np.std(pred, ddof=0))
    corr = (
        float(np.corrcoef(true, pred)[0, 1])
        if len(true) > 1 and true_std > 0 and pred_std > 0 else np.nan
    )
    return {
        "n": int(len(frame)),
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "r2": float(1.0 - np.sum(error ** 2) / sst) if sst > 0 else np.nan,
        "bias": float(np.mean(error)),
        "slope": (
            float(np.polyfit(true, pred, 1)[0])
            if len(true) > 1 and true_std > 0 else np.nan
        ),
        "corr": corr,
        "std_ratio": pred_std / true_std if true_std > 0 else np.nan,
    }


def metric_text(metric: dict) -> str:
    return (
        f"n={metric['n']:,}\n"
        f"MAE={metric['mae']:.2f} m\n"
        f"RMSE={metric['rmse']:.2f} m\n"
        f"R²={metric['r2']:.2f}\n"
        f"Bias={metric['bias']:+.2f} m\n"
        f"Slope={metric['slope']:.2f}\n"
        f"Corr={metric['corr']:.2f}\n"
        f"Std ratio={metric['std_ratio']:.2f}"
    )


def error_ylim(error):
    finite = np.asarray(error, dtype=float)
    finite = finite[np.isfinite(finite)]
    low, high = np.quantile(finite, [0.003, 0.997])
    return max(min(low * 1.30, -8.0), -45.0), min(max(high * 1.15, 6.0), 25.0)


def density_per_1m_cell(true, pred, axis_max):
    edges = np.arange(0.0, axis_max + 1.0001, 1.0)
    counts, _, _ = np.histogram2d(true, pred, bins=(edges, edges))
    ix = np.clip(np.searchsorted(edges, true, side="right") - 1, 0, len(edges) - 2)
    iy = np.clip(np.searchsorted(edges, pred, side="right") - 1, 0, len(edges) - 2)
    return counts[ix, iy]


def nice_scale(max_value, target_intervals=5):
    if not np.isfinite(max_value) or max_value <= 0:
        return 1.0, np.asarray([0, 1])
    raw = max_value / target_intervals
    magnitude = 10.0 ** np.floor(np.log10(raw))
    fraction = raw / magnitude
    nice = 1.0 if fraction <= 1 else 2.0 if fraction <= 2 else 5.0 if fraction <= 5 else 10.0
    step = max(1.0, nice * magnitude)
    upper = float(np.ceil(max_value / step) * step)
    return upper, np.arange(0.0, upper + 0.5 * step, step)


def plot_scatter(frame, forest, output_dir):
    cfg = SITES[forest]
    axis_max = cfg["eval_max"]
    true = frame["rh95"].to_numpy(float)
    pred = frame["prediction"].to_numpy(float)
    # Canopy height is physically non-negative. Clip only for graphical
    # display; compute_metrics(frame) below continues to use raw predictions.
    pred_plot = np.clip(pred, 0.0, axis_max)
    density = density_per_1m_cell(true, pred_plot, axis_max)
    order = np.argsort(density, kind="mergesort")
    upper, ticks = nice_scale(float(density.max()))
    metric = compute_metrics(frame)

    fig, ax = plt.subplots(figsize=(7.5, 7.0))
    points = ax.scatter(
        true[order], pred_plot[order], c=density[order],
        cmap="viridis", norm=Normalize(0.0, upper),
        s=float(np.clip(45000.0 / len(frame), 7.0, 16.0)),
        alpha=0.90, edgecolors="none", rasterized=True,
    )
    ax.plot([0, axis_max], [0, axis_max], "k--", linewidth=1.0)
    ax.set(xlim=(0, axis_max), ylim=(0, axis_max))
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("RH95 from GEDI waveforms (m)")
    ax.set_ylabel("Predicted height (m)")
    ax.grid(True, color="0.88", linewidth=0.55)
    ax.text(
        0.025, 0.975, metric_text(metric),
        transform=ax.transAxes, ha="left", va="top", fontsize=9,
        bbox={
            "facecolor": "white", "edgecolor": "0.72",
            "alpha": 0.70, "pad": 2.2,
        },
    )
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4.2%", pad=0.10)
    colorbar = fig.colorbar(points, cax=cax)
    colorbar.set_ticks(ticks)
    colorbar.set_ticklabels([f"{int(value):,}" for value in ticks])
    colorbar.ax.minorticks_off()
    harmonize_axis_typography(ax, colorbar.ax)
    export_figure(fig, output_dir, "01_scatter_observed_predicted")
    plt.show()
    plt.close(fig)


def class_errors(frame, edges):
    true = frame["rh95"].to_numpy(float)
    error = frame["prediction"].to_numpy(float) - true
    data, positions, labels, counts = [], [], [], []
    for index, (low, high) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (true >= low) & (true < high if index < len(edges) - 2 else true <= high)
        counts.append(int(mask.sum()))
        labels.append(f"{low:g}–{high:g}")
        if mask.any():
            data.append(error[mask])
            positions.append((low + high) / 2.0)
    return error, data, positions, labels, counts


def plot_height_distribution_error(frame, forest, output_dir):
    cfg = SITES[forest]
    true = frame["rh95"].to_numpy(float)
    pred = frame["prediction"].to_numpy(float)
    # Display classes use the conventional 0--5 m first label. The loaded
    # evaluation frame is already filtered to the frozen >=2 m domain, so
    # this relabelling does not add observations or change any metric.
    error, boxes, positions, _, _ = class_errors(frame, cfg["height_bins"])
    # Canopy height is physically non-negative. Clip only for graphical
    # display; the reported metrics continue to use raw predictions.
    pred_display = np.clip(pred, 0.0, cfg["eval_max"])
    hist_min = cfg["eval_min"]
    hist_edges = np.arange(hist_min, cfg["eval_max"] + 1.0001, 1.0)

    # Martin-style compact upper panel: same source width as panel (b),
    # but substantially shorter vertically.
    fig, count_ax = plt.subplots(figsize=(7.5, 2.80))
    count_ax.hist(true, bins=hist_edges, color=GEDI_HIST, alpha=0.92, label="GEDI Height")
    count_ax.hist(
        pred_display, bins=hist_edges,
        histtype="step", color=PRED_COLOR, linewidth=1.8, label="Predicted Height",
    )
    error_ax = count_ax.twinx()
    if boxes:
        error_ax.boxplot(
            boxes, positions=positions, widths=0.65, patch_artist=True,
            showfliers=False, manage_ticks=False,
            medianprops={"color": MEDIAN_COLOR, "linewidth": 1.6},
            boxprops={"facecolor": "black", "edgecolor": "black"},
            whiskerprops={"color": "black"}, capprops={"color": "black"},
        )
    error_ax.axhline(0, color="0.35", linestyle=":", linewidth=1.3)
    error_limits = error_ylim(error)
    error_ax.set_ylim(*error_limits)
    zero_fraction = (0.0 - error_limits[0]) / (error_limits[1] - error_limits[0])
    count_ax.set_ylabel("Count")
    error_ax.set_ylabel("Error (m)")
    # Keep the vertical title clear of the largest count tick labels.
    count_ax.yaxis.set_label_coords(-0.105, zero_fraction)
    error_ax.yaxis.set_label_coords(1.075, zero_fraction)
    # Display the physical CHM axis and conventional 5 m graduations from 0.
    count_ax.set_xlim(0.0, cfg["eval_max"])
    x_ticks = list(cfg["height_bins"])
    count_ax.set_xticks([x for x in x_ticks if 0.0 <= x <= cfg["eval_max"]])
    count_ax.set_xlabel("")  # Keep numeric ticks; omit redundant panel-(a) axis title.
    count_ax.legend(loc="upper right", framealpha=0.70)
    harmonize_axis_typography(count_ax, error_ax)
    export_figure(fig, output_dir, "02_height_distribution_and_error")
    plt.show()
    plt.close(fig)


def plot_height_error_bins(frame, forest, output_dir):
    cfg = SITES[forest]
    error, boxes, positions, labels, counts = class_errors(frame, cfg["height_bins"])
    centers = 0.5 * (cfg["height_bins"][:-1] + cfg["height_bins"][1:])
    fig, count_ax = plt.subplots(figsize=(10.5, 5.2))
    count_ax.bar(centers, counts, width=4.6, color=GEDI_BAR, alpha=0.82)
    error_ax = count_ax.twinx()
    if boxes:
        error_ax.boxplot(
            boxes, positions=positions, widths=1.85, patch_artist=True,
            showfliers=False, manage_ticks=False,
            medianprops={"color": MEDIAN_COLOR, "linewidth": 1.6},
            boxprops={"facecolor": "white", "edgecolor": "0.35"},
            whiskerprops={"color": "0.35"}, capprops={"color": "0.35"},
        )
    error_ax.axhline(0, color=ACCENT_BLUE, linestyle="--", linewidth=1.1)
    error_ax.set_ylim(*error_ylim(error))
    count_ax.set_xlim(0, cfg["eval_max"])
    count_ax.set_xticks(centers, labels)
    count_ax.set_xlabel("")
    export_figure(fig, output_dir, "03_height_class_errors")
    plt.show()
    plt.close(fig)


def plot_signed_error(frame, forest, output_dir):
    error = frame["prediction"].to_numpy(float) - frame["rh95"].to_numpy(float)
    fig, ax = plt.subplots(figsize=(7.8, 4.5))
    ax.hist(error, bins=45, alpha=0.85, color=GEDI_BAR, edgecolor="white")
    ax.axvline(0, color="black", linestyle="--", linewidth=1.1)
    ax.axvline(np.median(error), color=MEDIAN_COLOR, linewidth=1.7,
               label=f"Median={np.median(error):+.2f} m")
    ax.axvline(np.mean(error), color=ACCENT_ORANGE, linestyle="--", linewidth=1.4,
               label=f"Mean={np.mean(error):+.2f} m")
    ax.set_xlabel("Prediction − GEDI (m)")
    ax.legend(framealpha=0.70)
    export_figure(fig, output_dir, "04_signed_error_distribution")
    plt.show()
    plt.close(fig)


def plot_residuals(frame, forest, output_dir):
    cfg = SITES[forest]
    true = frame["rh95"].to_numpy(float)
    error, _, _, _, _ = class_errors(frame, cfg["height_bins"])
    centers, medians = [], []
    for index, (low, high) in enumerate(zip(cfg["height_bins"][:-1], cfg["height_bins"][1:])):
        mask = (true >= low) & (true < high if index < len(cfg["height_bins"]) - 2 else true <= high)
        if mask.any():
            centers.append((low + high) / 2.0)
            medians.append(float(np.median(error[mask])))
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    ax.scatter(true, error, s=9, alpha=0.28, linewidths=0, color=ACCENT_BLUE, rasterized=True)
    ax.axhline(0, color="black", linestyle="--", linewidth=1.1)
    ax.plot(centers, medians, color=MEDIAN_COLOR, linewidth=1.8, marker="o",
            label="Median residual")
    ax.set_xlim(0, cfg["eval_max"])
    ax.set_ylim(*error_ylim(error))
    ax.set_xlabel("RH95 from GEDI waveforms (m)")
    ax.legend(framealpha=0.70)
    export_figure(fig, output_dir, "05_residuals_vs_height")
    plt.show()
    plt.close(fig)


def plot_absolute_error_cdf(frame, forest, output_dir):
    absolute = np.sort(np.abs(
        frame["prediction"].to_numpy(float) - frame["rh95"].to_numpy(float)
    ))
    cumulative = np.arange(1, len(absolute) + 1) / len(absolute)
    fig, ax = plt.subplots(figsize=(7.8, 4.6))
    ax.plot(absolute, cumulative, color=ACCENT_ORANGE, linewidth=2.1)
    for threshold in (1, 2, 3, 5, 10):
        fraction = 100.0 * float(np.mean(absolute <= threshold))
        ax.axvline(threshold, color="0.4", linestyle="--", linewidth=0.8, alpha=0.55)
        ax.text(threshold, 0.035, f"{fraction:.0f}% ≤ {threshold} m",
                rotation=90, va="bottom", ha="right", fontsize=8)
    ax.set_xlim(0, max(float(np.percentile(absolute, 99.5)), 5.0))
    ax.set_ylim(0, 1.01)
    ax.set_xlabel("|Prediction − GEDI| (m)")
    export_figure(fig, output_dir, "06_absolute_error_cdf")
    plt.show()
    plt.close(fig)


def run_forest_report(forest: str) -> dict:
    frame = load_phase2_test(forest)
    cfg = SITES[forest]
    output_dir = OUTPUT_ROOT / cfg["ecosystem"].replace(" ", "_") / forest
    metric = compute_metrics(frame)
    print(f"{forest} | Phase 2 TEST | n={len(frame):,} | output={output_dir}")
    plot_scatter(frame, forest, output_dir)
    plot_height_distribution_error(frame, forest, output_dir)
    plot_height_error_bins(frame, forest, output_dir)
    plot_signed_error(frame, forest, output_dir)
    plot_residuals(frame, forest, output_dir)
    plot_absolute_error_cdf(frame, forest, output_dir)
    pd.DataFrame([metric]).assign(
        forest=forest,
        ecosystem=cfg["ecosystem"],
        eval_min_m=cfg["eval_min"],
        eval_max_m=cfg["eval_max"],
        prediction_column="pred_on_growthloss",
    ).to_csv(output_dir / "metrics.csv", index=False)
    return {"forest": forest, "ecosystem": cfg["ecosystem"], **metric}


# Chapter 1 — Moderately Dense: Ifran Forest

The frozen Phase 2 prediction is evaluated on the canonical unique-nearest TEST
support within the 2–45 m GEDI RH95 domain.

In [ ]:
ifran_metrics = run_forest_report("Ifran")

# Chapter 2 — Low Density: Maamoura Forest

The frozen Phase 2 prediction is evaluated on the canonical unique-nearest TEST
support within the 2–20 m GEDI RH95 domain.

In [ ]:
maamoura_metrics = run_forest_report("Maamoura")

# Chapter 3 — Sparse: Agadir Forest

The frozen Phase 2 prediction is evaluated on the canonical unique-nearest TEST
support within the 2–20 m GEDI RH95 domain.

In [ ]:
agadir_metrics = run_forest_report("Agadir")

# Final TEST summary

This table is an audit of the three frozen Phase 2 predictions used by all
figures. It is not used to select a checkpoint.

In [ ]:
phase2_plot_metrics = pd.DataFrame([
    ifran_metrics,
    maamoura_metrics,
    agadir_metrics,
])
summary_path = OUTPUT_ROOT / "phase2_three_forests_plot_metrics.csv"
phase2_plot_metrics.to_csv(summary_path, index=False)
display(phase2_plot_metrics.round(4))
print("Saved:", summary_path)
print("[PASS] Plots use pred_on_growthloss only.")
print("[PASS] Unique-nearest TEST shots are used exactly once.")
print("[PASS] No training, inference or checkpoint selection was run.")